In [3]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from itertools import product
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy


# --------------------------------------------------
# General settings
# --------------------------------------------------

DATA_DIR = Path("Datasets")

DATASETS = [20, 30, 40]

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

N_FOLDS = 5
SEEDS = [0, 1, 2]

BATCH_SIZE = 256
MAX_EPOCHS = 150
PATIENCE = 20

SPLIT_SEED = 0

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)


# --------------------------------------------------
# Hyperparameter search space
# --------------------------------------------------

LEARNING_RATES = [3e-3, 1e-3, 3e-4]
WEIGHT_DECAYS = [0, 1e-3, 1e-1]
DROPOUT_RATES = [0.0, 0.1, 0.2]


# --------------------------------------------------
# Six architectures
# --------------------------------------------------

ARCHITECTURES = {
    "32": (32,),
  #  "64": (64,),
    "32-32": (32, 32),
   # "64-64": (64, 64),
    #"32-32-32": (32, 32, 32),
    "64-64-64": (64, 64, 64),
}


def make_mlp(input_dim, widths, dropout):
    layers = []
    h = input_dim

    for width in widths:
        layers.extend([
            nn.Linear(h, width),
            nn.ReLU(),
            nn.Dropout(dropout)
        ])
        h = width

    layers.append(nn.Linear(h, 1))

    return nn.Sequential(*layers)


def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

Device: cpu


In [4]:
# --------------------------------------------------
# Load all datasets and create initial splits
# --------------------------------------------------

data = {}

for N in DATASETS:

    X = np.load(DATA_DIR / f"kryptonite-{N}-X.npy")
    y = np.load(DATA_DIR / f"kryptonite-{N}-y.npy")

    # Preserve the -1, 0, 1 input values
    X = X.astype(np.float32)

    # Convert labels to 0 and 1 if they are -1 and 1
    y = y.reshape(-1)

    unique_labels = np.unique(y)

    if set(unique_labels).issubset({-1, 1}):
        y = (y == 1).astype(np.float32)
    else:
        y = y.astype(np.float32)

    # Initial 80/20 split
    X_dev, X_test, y_dev, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=SPLIT_SEED
    )

    data[N] = {
        "X_dev": X_dev,
        "y_dev": y_dev,
        "X_test": X_test,
        "y_test": y_test
    }

    print(
        f"Kryptonite-{N}: "
        f"development={len(X_dev)}, "
        f"test={len(X_test)}"
    )

Kryptonite-20: development=9600, test=2400
Kryptonite-30: development=16000, test=4000
Kryptonite-40: development=25600, test=6400


In [5]:
# --------------------------------------------------
# Training function
# --------------------------------------------------

def train_model(
    X_train,
    y_train,
    X_val,
    y_val,
    widths,
    learning_rate,
    weight_decay,
    dropout,
    seed,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE
):

    # Reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    X_train_t = torch.tensor(
        X_train, dtype=torch.float32
    ).to(DEVICE)

    y_train_t = torch.tensor(
        y_train.reshape(-1, 1), dtype=torch.float32
    ).to(DEVICE)

    X_val_t = torch.tensor(
        X_val, dtype=torch.float32
    ).to(DEVICE)

    y_val_t = torch.tensor(
        y_val.reshape(-1, 1), dtype=torch.float32
    ).to(DEVICE)

    train_dataset = TensorDataset(
        X_train_t, y_train_t
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    model = make_mlp(
        input_dim=X_train.shape[1],
        widths=widths,
        dropout=dropout
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.BCEWithLogitsLoss()

    best_accuracy = 0.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):

        model.train()

        for xb, yb in train_loader:

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

        # Validation
        model.eval()

        with torch.no_grad():
            val_logits = model(X_val_t)
            val_predictions = (
                val_logits >= 0
            ).float()

            accuracy = (
                val_predictions == y_val_t
            ).float().mean().item()

        if accuracy > best_accuracy:

            best_accuracy = accuracy
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)

    return {
        "val_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "parameters": count_parameters(model),
        "state_dict": best_state
    }

In [6]:
# --------------------------------------------------
# Hyperparameter search
# --------------------------------------------------

hyperparameter_results = []
best_hyperparameters = {}

for N in DATASETS:

    print(f"\nHyperparameter search: Kryptonite-{N}")

    X_dev = data[N]["X_dev"]
    y_dev = data[N]["y_dev"]

    # Temporary train/validation split within development data
    X_train, X_val, y_train, y_val = train_test_split(
        X_dev,
        y_dev,
        test_size=VALIDATION_SIZE,
        stratify=y_dev,
        random_state=SPLIT_SEED
    )

    # Use a fixed architecture for hyperparameter selection
    search_architecture = (32, 32, 32)

    best_score = -1
    best_config = None

    combinations = list(product(
        LEARNING_RATES,
        WEIGHT_DECAYS,
        DROPOUT_RATES
    ))

    for lr, wd, dropout in combinations:

        seed_scores = []
        seed_epochs = []

        for seed in SEEDS:

            result = train_model(
                X_train,
                y_train,
                X_val,
                y_val,
                widths=search_architecture,
                learning_rate=lr,
                weight_decay=wd,
                dropout=dropout,
                seed=seed
            )

            seed_scores.append(result["val_accuracy"])
            seed_epochs.append(result["best_epoch"])

        mean_score = np.mean(seed_scores)

        hyperparameter_results.append({
            "N": N,
            "learning_rate": lr,
            "weight_decay": wd,
            "dropout": dropout,
            "mean_val_accuracy": mean_score,
            "std_val_accuracy": np.std(seed_scores, ddof=1),
            "mean_best_epoch": np.mean(seed_epochs)
        })

        print(
            f"lr={lr}, wd={wd}, dropout={dropout} "
            f"| validation accuracy={mean_score:.4f}"
        )

        if mean_score > best_score:

            best_score = mean_score

            best_config = {
                "learning_rate": lr,
                "weight_decay": wd,
                "dropout": dropout
            }

    best_hyperparameters[N] = best_config

    print("\nSelected hyperparameters:", best_config)
    print(f"Best validation accuracy: {best_score:.4f}")


hyperparameter_df = pd.DataFrame(hyperparameter_results)

print("\nHYPERPARAMETER SEARCH SUMMARY")
print(
    hyperparameter_df
    .sort_values(["N", "mean_val_accuracy"], ascending=[True, False])
    .to_string(index=False)
)


Hyperparameter search: Kryptonite-20
lr=0.003, wd=0, dropout=0.0 | validation accuracy=0.9273
lr=0.003, wd=0, dropout=0.1 | validation accuracy=0.9562
lr=0.003, wd=0, dropout=0.2 | validation accuracy=0.9491
lr=0.003, wd=0.001, dropout=0.0 | validation accuracy=0.9285
lr=0.003, wd=0.001, dropout=0.1 | validation accuracy=0.9533
lr=0.003, wd=0.001, dropout=0.2 | validation accuracy=0.9413
lr=0.003, wd=0.1, dropout=0.0 | validation accuracy=0.9337
lr=0.003, wd=0.1, dropout=0.1 | validation accuracy=0.9575
lr=0.003, wd=0.1, dropout=0.2 | validation accuracy=0.9470
lr=0.001, wd=0, dropout=0.0 | validation accuracy=0.9104
lr=0.001, wd=0, dropout=0.1 | validation accuracy=0.9568
lr=0.001, wd=0, dropout=0.2 | validation accuracy=0.9278
lr=0.001, wd=0.001, dropout=0.0 | validation accuracy=0.9146
lr=0.001, wd=0.001, dropout=0.1 | validation accuracy=0.9535
lr=0.001, wd=0.001, dropout=0.2 | validation accuracy=0.9328
lr=0.001, wd=0.1, dropout=0.0 | validation accuracy=0.9238
lr=0.001, wd=0.1, 

In [7]:
# --------------------------------------------------
# Architecture cross-validation
# --------------------------------------------------

cv_results = []

for N in DATASETS:

    print(f"\nArchitecture CV: Kryptonite-{N}")

    X_dev = data[N]["X_dev"]
    y_dev = data[N]["y_dev"]

    config = best_hyperparameters[N]

    skf = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SPLIT_SEED
    )

    folds = list(
        skf.split(X_dev, y_dev.astype(int))
    )

    for architecture_name, widths in ARCHITECTURES.items():

        print(f"\nArchitecture: {architecture_name}")

        for fold, (train_idx, val_idx) in enumerate(
            folds, start=1
        ):

            X_train = X_dev[train_idx]
            y_train = y_dev[train_idx]

            X_val = X_dev[val_idx]
            y_val = y_dev[val_idx]

            for seed in SEEDS:

                result = train_model(
                    X_train,
                    y_train,
                    X_val,
                    y_val,
                    widths=widths,
                    learning_rate=config["learning_rate"],
                    weight_decay=config["weight_decay"],
                    dropout=config["dropout"],
                    seed=seed
                )

                cv_results.append({
                    "N": N,
                    "architecture": architecture_name,
                    "fold": fold,
                    "seed": seed,
                    "accuracy": result["val_accuracy"],
                    "best_epoch": result["best_epoch"],
                    "parameters": result["parameters"]
                })

                print(
                    f"Fold {fold}, seed {seed}: "
                    f"{result['val_accuracy']:.4f}"
                )


cv_df = pd.DataFrame(cv_results)


Architecture CV: Kryptonite-20

Architecture: 32
Fold 1, seed 0: 0.7214
Fold 1, seed 1: 0.7026
Fold 1, seed 2: 0.6990
Fold 2, seed 0: 0.6932
Fold 2, seed 1: 0.6812
Fold 2, seed 2: 0.6880
Fold 3, seed 0: 0.7141
Fold 3, seed 1: 0.7266
Fold 3, seed 2: 0.7156
Fold 4, seed 0: 0.6698
Fold 4, seed 1: 0.6896
Fold 4, seed 2: 0.6911
Fold 5, seed 0: 0.7073
Fold 5, seed 1: 0.6740
Fold 5, seed 2: 0.7089

Architecture: 32-32
Fold 1, seed 0: 0.9172
Fold 1, seed 1: 0.9458
Fold 1, seed 2: 0.9370
Fold 2, seed 0: 0.9297
Fold 2, seed 1: 0.9286
Fold 2, seed 2: 0.9208
Fold 3, seed 0: 0.9286
Fold 3, seed 1: 0.9307
Fold 3, seed 2: 0.9464
Fold 4, seed 0: 0.9417
Fold 4, seed 1: 0.9281
Fold 4, seed 2: 0.9349
Fold 5, seed 0: 0.9229
Fold 5, seed 1: 0.9182
Fold 5, seed 2: 0.9370

Architecture: 64-64-64
Fold 1, seed 0: 0.9589
Fold 1, seed 1: 0.9651
Fold 1, seed 2: 0.9625
Fold 2, seed 0: 0.9505
Fold 2, seed 1: 0.9510
Fold 2, seed 2: 0.9438
Fold 3, seed 0: 0.9521
Fold 3, seed 1: 0.9474
Fold 3, seed 2: 0.9500
Fold 4, 

In [8]:
# --------------------------------------------------
# CV summary
# --------------------------------------------------

fold_means = (
    cv_df
    .groupby(["N", "architecture", "fold"])
    .agg(
        fold_accuracy=("accuracy", "mean"),
        fold_best_epoch=("best_epoch", "median"),
        parameters=("parameters", "first")
    )
    .reset_index()
)

cv_summary = (
    fold_means
    .groupby(["N", "architecture"])
    .agg(
        mean_cv_accuracy=("fold_accuracy", "mean"),
        std_cv_accuracy=("fold_accuracy", "std"),
        median_best_epoch=("fold_best_epoch", "median"),
        parameters=("parameters", "first")
    )
    .reset_index()
    .sort_values(
        ["N", "mean_cv_accuracy"],
        ascending=[True, False]
    )
)

print("\nCROSS-VALIDATION RESULTS")
print(cv_summary.round(4).to_string(index=False))


# Select the best architecture separately for each dataset
selected_architectures = {}

for N in DATASETS:

    results_N = cv_summary[
        cv_summary["N"] == N
    ]

    best_row = results_N.iloc[0]

    selected_architectures[N] = {
        "architecture": best_row["architecture"],
        "widths": ARCHITECTURES[best_row["architecture"]],
        "median_best_epoch": int(
            best_row["median_best_epoch"]
        ),
        "parameters": int(best_row["parameters"])
    }

    print(
        f"\nKryptonite-{N}: "
        f"selected {best_row['architecture']}, "
        f"CV accuracy={best_row['mean_cv_accuracy']:.4f}, "
        f"parameters={best_row['parameters']}"
    )


CROSS-VALIDATION RESULTS
 N architecture  mean_cv_accuracy  std_cv_accuracy  median_best_epoch  parameters
20     64-64-64            0.9519           0.0063               48.0        9729
20        32-32            0.9312           0.0046              140.0        1761
20           32            0.6988           0.0145              132.0         705
30     64-64-64            0.9569           0.0021               47.0       10369
30        32-32            0.9515           0.0040               95.0        2081
30           32            0.6880           0.0101              120.0        1025
40     64-64-64            0.9582           0.0026               32.0       11009
40        32-32            0.9535           0.0039               80.0        2401
40           32            0.7029           0.0163              140.0        1345

Kryptonite-20: selected 64-64-64, CV accuracy=0.9519, parameters=9729

Kryptonite-30: selected 64-64-64, CV accuracy=0.9569, parameters=10369

Kryptonite

In [9]:
# --------------------------------------------------
# Fixed-epoch final training function
# --------------------------------------------------

def train_fixed_epochs(
    X_train,
    y_train,
    widths,
    learning_rate,
    weight_decay,
    dropout,
    seed,
    epochs
):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    X_train_t = torch.tensor(
        X_train, dtype=torch.float32
    ).to(DEVICE)

    y_train_t = torch.tensor(
        y_train.reshape(-1, 1), dtype=torch.float32
    ).to(DEVICE)

    train_dataset = TensorDataset(
        X_train_t, y_train_t
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    model = make_mlp(
        input_dim=X_train.shape[1],
        widths=widths,
        dropout=dropout
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):

        model.train()

        for xb, yb in train_loader:

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

    return model


# --------------------------------------------------
# Final held-out test evaluation
# --------------------------------------------------

final_results = []

for N in DATASETS:

    print(f"\nFinal evaluation: Kryptonite-{N}")

    X_dev = data[N]["X_dev"]
    y_dev = data[N]["y_dev"]

    X_test = data[N]["X_test"]
    y_test = data[N]["y_test"]

    selection = selected_architectures[N]
    config = best_hyperparameters[N]

    widths = selection["widths"]
    epochs = selection["median_best_epoch"]

    test_X_t = torch.tensor(
        X_test, dtype=torch.float32
    ).to(DEVICE)

    test_y_t = torch.tensor(
        y_test.reshape(-1, 1), dtype=torch.float32
    ).to(DEVICE)

    test_scores = []

    for seed in [10, 20, 30, 40, 50]:

        model = train_fixed_epochs(
            X_dev,
            y_dev,
            widths=widths,
            learning_rate=config["learning_rate"],
            weight_decay=config["weight_decay"],
            dropout=config["dropout"],
            seed=seed,
            epochs=epochs
        )

        model.eval()

        with torch.no_grad():

            logits = model(test_X_t)
            predictions = (logits >= 0).float()

            accuracy = (
                predictions == test_y_t
            ).float().mean().item()

        test_scores.append(accuracy)

        print(f"Seed {seed}: test accuracy={accuracy:.4f}")

    mean_test = np.mean(test_scores)
    std_test = np.std(test_scores, ddof=1)

    final_results.append({
        "N": N,
        "architecture": selection["architecture"],
        "parameters": selection["parameters"],
        "learning_rate": config["learning_rate"],
        "weight_decay": config["weight_decay"],
        "dropout": config["dropout"],
        "epochs": epochs,
        "mean_test_accuracy": mean_test,
        "std_test_accuracy": std_test
    })

    print(
        f"\nKryptonite-{N} final test accuracy: "
        f"{mean_test:.4f} ± {std_test:.4f}"
    )


final_df = pd.DataFrame(final_results)

print("\nFINAL HELD-OUT TEST RESULTS")
print(final_df.round(4).to_string(index=False))


Final evaluation: Kryptonite-20
Seed 10: test accuracy=0.9529
Seed 20: test accuracy=0.9529
Seed 30: test accuracy=0.9500
Seed 40: test accuracy=0.9567
Seed 50: test accuracy=0.9525

Kryptonite-20 final test accuracy: 0.9530 ± 0.0024

Final evaluation: Kryptonite-30
Seed 10: test accuracy=0.9560
Seed 20: test accuracy=0.9572
Seed 30: test accuracy=0.9560
Seed 40: test accuracy=0.9542
Seed 50: test accuracy=0.9563

Kryptonite-30 final test accuracy: 0.9559 ± 0.0011

Final evaluation: Kryptonite-40
Seed 10: test accuracy=0.9559
Seed 20: test accuracy=0.9539
Seed 30: test accuracy=0.9552
Seed 40: test accuracy=0.9581
Seed 50: test accuracy=0.9581

Kryptonite-40 final test accuracy: 0.9563 ± 0.0019

FINAL HELD-OUT TEST RESULTS
 N architecture  parameters  learning_rate  weight_decay  dropout  epochs  mean_test_accuracy  std_test_accuracy
20     64-64-64        9729          0.003           0.1      0.1      48              0.9530             0.0024
30     64-64-64       10369          0.0